# Vector Memory with ChromaDB

*Notebook 21*

What happens when you need to retrieve facts by meaning, not exact words?

SQLite works well for structured lookups, but when users phrase things differently than how memories were stored, exact matching fails.

Vector memory solves this with semantic retrieval.
<br>
<br>

**Topics:**

- What vector databases are and how they differ from SQLite

- Embeddings — storing meaning, not just text

- Semantic retrieval — finding relevant memories by concept

- Building a vector memory store with ChromaDB

- When to use vector memory vs SQLite

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

# Notebook-specific imports
import chromadb

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

SQLite fails when the user's phrasing doesn't match what was stored.

Vector memory retrieves by meaning — "what flavors does he like?" finds "Alex enjoys spicy food" even though none of the words match.

---

## 🧬 Part 1: How Vector Memory Works

Standard databases find exact matches. Vector databases find meaning matches using embeddings — mathematical representations of text.

1. **Embed:** A model converts text into a list of numbers (a vector)

2. **Store:** The vector database saves those numbers

3. **Query:** The user's question is also converted to a vector

4. **Match:** The database finds stored vectors closest in meaning to the query vector

---

## 🔍 Part 2: Why Keyword Search Fails

Exact keyword matching fails on paraphrased queries.

In [ ]:
# Exact match simulation — simple keyword lookup
memories = [
    "Alex prefers Python for data science and JavaScript for web apps.",
    "The company uses Slack for internal chat and Zoom for meetings.",
    "The production servers are located in the US-East region.",
    "Standard shipping takes 3-5 business days."
]

query = "What coding tools does the developer prefer?"

# Simple keyword search — looks for exact word overlap
exact_matches = [m for m in memories if any(
    word.lower() in m.lower()
    for word in query.split()
    if len(word) > 4
)]

print(f"🔍 Query: {query}")
print(f"Exact match results: {exact_matches if exact_matches else 'No matches found'}")
print("\n💡 'coding tools' and 'developer' didn't match 'Python' or 'JavaScript' — different words, same concept.")

Vector retrieval finds the match because it compares meaning, not words.

---

## 🛠️ Part 3: Building a Memory Store with ChromaDB

ChromaDB is a popular, lightweight vector database that runs entirely in your notebook.

In [ ]:
# 1. Initialize Chroma client (in-memory for this demo)
chroma_client = chromadb.EphemeralClient()

# 2. Create a collection to store our agent's memories
collection = chroma_client.get_or_create_collection(name="agent_memories")

# 3. Add some memories
collection.add(
    documents=[
        "Alex prefers Python for data science and JavaScript for web apps.",
        "The company uses Slack for internal chat and Zoom for meetings.",
        "The production servers are located in the US-East region.",
        "Standard shipping takes 3-5 business days."
    ],
    ids=["id1", "id2", "id3", "id4"]  # each document needs a unique ID; use it later to update or delete a specific memory
)

print("✅ Vector memory collection created and populated.")

For a real project, swap `EphemeralClient()` for `chromadb.PersistentClient(path='./chroma_db')` — the API is identical and the collection survives restarts. Ephemeral is used here only to keep the demo self-contained.

In this demo, Chroma uses a small local embedding model that ships with the library — no API calls, no cost.

For production you can swap in OpenAI's embedding model for higher quality — the rest of the code stays the same.

In your own project, you usually change only the collection name and the texts you store — Chroma handles the embedding step for both storage and search.

# Same intent as Part 2's query — different words — semantic search succeeds where exact match failed
query = "What are the preferred programming tools?"

results = collection.query(
    query_texts=[query],
    n_results=1
)
# results is a dict with 'documents', 'ids', and 'distances'; each is a list-of-lists, one inner list per query

retrieved_memory = results['documents'][0][0]

print(f"🔍 Query: {query}")
print(f"🧠 Retrieved: {retrieved_memory}")

### 💡 Why This Works

The query does not need to match exact words.

Embeddings let Chroma retrieve memories that are semantically related to the user's wording.

## 🤖 Part 4: Connecting Memory to an Agent

Reach for this pattern when the agent only occasionally needs old notes, docs, or preferences — sending the full memory store on every turn would waste tokens and clutter the prompt.

To use this in an agent, we create a tool that queries our vector database.

This keeps the agent's prompt clean while giving it on-demand access to a massive history.

@function_tool
def search_past_memories(query: str) -> str:
    """Search long-term memory for relevant facts or preferences."""
    print(f"[Memory] Querying: {query}")
    results = collection.query(query_texts=[query], n_results=2)
    memories = results['documents'][0]
    return "Relevant memories: " + "; ".join(memories)

memory_agent_instructions = (
    "You are a helpful assistant.\n"
    "Use the search_past_memories tool to look up context if needed."
)

# --------------------------------------------------------------
memory_agent = Agent(
    name="MemoryAssistant",
    instructions=memory_agent_instructions,
    model=MODEL,
    tools=[search_past_memories]
)

# --------------------------------------------------------------
print("✅ Memory agent ready")

result = await Runner.run(memory_agent, input="Based on what you know about Alex, what tech stack should I recommend for his data science project?")

print(result.final_output)

### 💡 Why This Works

The agent calls `search_past_memories` as a tool — it queries the vector store at runtime rather than receiving everything upfront.

Only the most relevant memories reach the prompt, keeping context lean regardless of how large the memory store grows.

⚠️ **Security note:** Retrieved vector documents are unverified context — if memory content comes from user input or external sources, it could contain adversarial text.

Prompt injection defenses are covered in Lesson 23.

## 🧭 Part 5: When to Use Vector Memory vs SQLite

Both stores persist data — but they answer different kinds of questions.

<div style="text-align: left; display: inline-block;">

| | SQLite memory | Vector Memory |
|---|---|---|
| Retrieval method | Structured or exact lookup | Semantic similarity |
| Best for | Facts, preferences, structured history | Large knowledge bases, freeform notes |
| Query flexibility | Low — must match session exactly | High — finds meaning across phrasings |
| Setup complexity | None (built into SDK) | Requires ChromaDB or similar |

</div>

Use SQLite when you need reliable exact-match recall of structured facts.

Use vector memory when you need to retrieve relevant context from a large, freeform store where the user's phrasing may not match what was stored.

---

## 💪 Practice Exercises

### Exercise 1: Build a Travel Preferences Store

*Covers: vector memory — semantic retrieval with varied phrasing*

Build a vector store of travel preferences and query it with different phrasing to confirm semantic retrieval works.

**Hint:** Store preferences using one set of words (e.g. "I enjoy cold mountain air") and query with different wording (e.g. "warm or cool destinations?") to confirm semantic retrieval is working.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Build a Travel Preferences Store
# --------------------------------------------------------------
# Objective: Build a vector store of travel preferences and query it
#            with different phrasing to confirm semantic retrieval works.

# TODO 1: Create a new Chroma collection called "travel_prefs"

# TODO 2: Add 3-4 diverse preferences

# TODO 3: Create a tool that queries this collection

# TODO 4: Run an agent that searches memory and suggests a specific city

# --- Write your code below this line ---

---

### Exercise 2: SQLite or Vector?

*Covers: memory selection — choosing the right storage approach*

For each scenario below, decide whether SQLite or vector memory is the better fit — then implement your choice for one of them.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: SQLite or Vector?
# --------------------------------------------------------------
# Objective: Choose the right memory store for each scenario,
#            then implement one.
#
# Scenario A: Remember a user's name, timezone, and preferred language
# Scenario B: Store meeting notes and retrieve relevant ones by topic
# Scenario C: Track a user's explicit yes/no preferences for a recommendation engine
# Scenario D: Store a product FAQ and retrieve answers by natural language question
#
# TODO 1: For each scenario, write a comment: # Best fit: SQLite or Vector? Why?
#
# TODO 2: Pick one scenario and implement it:
#         - If SQLite: create a SQLiteSession and run 2 turns
#         - If Vector: create a ChromaDB collection, add entries, and query it

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Semantic Retrieval:**

- Vector memory finds information by concept, not just keywords

- Essential for large datasets where you can't fit everything in the prompt
<br>
<br>

**Embeddings:**

- Turning text into numbers allows for mathematical meaning comparisons

- ChromaDB manages the embedding and comparison process automatically
<br>
<br>

**When to Use Each:**

- SQLite: best for exact facts, structured data, and small session histories

- Vector memory: best for large knowledge bases, finding conceptual links, and retrieving relevant context at scale

---

## 📍 Next Step

**Notebook 22: Guardrails**  

Now that your agent has memory, learn how to prevent it from using that memory in dangerous or off-topic ways.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-21-vector-memory)

---